# 05 — Ablation Study: YOLO26-cls **with vs without C2PSA**

## Research Question
> *Apakah blok attention C2PSA dalam YOLO26 memberikan kontribusi signifikan untuk klasifikasi aksara Jawa?*

## Setup
Dua variant identik kecuali satu modul:

| Variant | Layer terakhir backbone | Param count |
|---|---|---|
| **A: yolo26-attn** | `C2PSA` (Position-Sensitive Spatial Attention) | 1,556,724 |
| **B: yolo26-noattn** | `C3k2` (no attention, struktur biasa) | 1,653,108 |

Param count di atas untuk nc=20 (20 kelas aksara Jawa), konsisten dengan Tabel 2 paper.

**Perhatian**: kedua variant di-train **from scratch** (tanpa pretrained ImageNet) untuk fair comparison, karena modified architecture tidak bisa pakai bobot pretrained YOLO26 standar.

## Protokol
- 5 seeds: `[42, 123, 7, 2024, 999]`
- 50 epochs, patience 5
- imgsz 128, batch 64, workers 4
- Same augmentation, same optimizer (AdamW auto-set)

## Hipotesis
**H0**: Tidak ada perbedaan signifikan akurasi antara variant A dan B (p ≥ 0.05)\
**H1**: Ada perbedaan signifikan (p < 0.05)

Jika H0 diterima → kontribusi paper: *"attention block tidak diperlukan untuk klasifikasi aksara Jawa"*.

In [1]:
import time, gc
from pathlib import Path
import torch
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from ultralytics import YOLO

ROOT = Path('D:/Skripsi/AksaraJawa')
DATA = ROOT / 'DatasetPhiard'
RUNS_AB = ROOT / 'runs' / 'ablation_c2psa_clean'
RUNS_AB.mkdir(parents=True, exist_ok=True)

YAML_ATTN   = str(ROOT / 'configs' / 'yolo26-cls-attn.yaml')   # akan di-buat dari original
YAML_NOATTN = str(ROOT / 'configs' / 'yolo26-cls-noattn.yaml')

# Quick sanity check
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))

CUDA: True | NVIDIA GeForce RTX 5080 Laptop GPU


## Setup config — copy original YAML untuk fair comparison (both from scratch)

In [2]:
# Copy original YOLO26-cls.yaml ke configs/ kita biar dua-duanya konsisten dipakai dari YAML
import shutil, ultralytics
src_yaml = Path(ultralytics.__file__).parent / 'cfg' / 'models' / '26' / 'yolo26-cls.yaml'
dst_yaml = Path(YAML_ATTN)
if not dst_yaml.exists():
    shutil.copy(src_yaml, dst_yaml)
print(f'YAML attn:   {dst_yaml}  exists={dst_yaml.exists()}')
print(f'YAML noattn: {YAML_NOATTN}  exists={Path(YAML_NOATTN).exists()}')

# Verify param counts
# Catatan: angka di bawah memakai nc default YAML. Saat training dengan 20
# kelas aksara Jawa, head menyesuaikan ke nc=20, param final = 1,556,724
# (attn) dan 1,653,108 (noattn) -> lihat Tabel 2 paper.
for label, yml in [('with C2PSA   ', YAML_ATTN), ('without C2PSA', YAML_NOATTN)]:
    m = YOLO(yml)
    n = sum(p.numel() for p in m.model.parameters())
    print(f'  {label}: {n:,} params')
    del m

YAML attn:   D:\Skripsi\AksaraJawa\configs\yolo26-cls-attn.yaml  exists=True
YAML noattn: D:\Skripsi\AksaraJawa\configs\yolo26-cls-noattn.yaml  exists=True
WARNING no model scale passed. Assuming scale='n'.
YOLO26-cls-attn summary: 86 layers, 2,812,104 parameters, 2,812,104 gradients, 4.3 GFLOPs
  with C2PSA   : 2,812,104 params
WARNING no model scale passed. Assuming scale='n'.
YOLO26-cls-noattn summary: 86 layers, 2,908,488 parameters, 2,908,488 gradients, 4.4 GFLOPs
  without C2PSA: 2,908,488 params


## Multi-seed training kedua variant

10 run training (5 seeds × 2 variants) ~12-15 menit di RTX 5080.

In [3]:
SEEDS = [42, 123, 7, 2024, 999]
results = []

def train_variant(yaml_path, tag, seed):
    m = YOLO(yaml_path)
    t0 = time.time()
    m.train(
        data=str(DATA), epochs=50, imgsz=128, batch=64, device=0, workers=4,
        project=str(RUNS_AB), name=f'{tag}_seed{seed}', exist_ok=True,
        patience=5, plots=False, seed=seed, verbose=False,
    )
    dt = time.time() - t0
    val = m.val(verbose=False)
    out = {'variant': tag, 'seed': seed,
           'top1': float(val.top1), 'top5': float(val.top5),
           'time_s': dt}
    del m, val
    gc.collect(); torch.cuda.empty_cache()
    return out

print('Mulai ablation training...\n')
for seed in SEEDS:
    print(f'=== seed={seed} ===')
    r1 = train_variant(YAML_ATTN, 'attn', seed)
    print(f'  with    C2PSA: top1={r1["top1"]:.4f}  time={r1["time_s"]:.1f}s')
    r2 = train_variant(YAML_NOATTN, 'noattn', seed)
    print(f'  without C2PSA: top1={r2["top1"]:.4f}  time={r2["time_s"]:.1f}s')
    results += [r1, r2]

print('\nAblation training selesai.')

Mulai ablation training...

=== seed=42 ===
WARNING no model scale passed. Assuming scale='n'.
YOLO26-cls-attn summary: 86 layers, 2,812,104 parameters, 2,812,104 gradients, 4.3 GFLOPs
Ultralytics 8.4.46  Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080 Laptop GPU, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Skripsi\AksaraJawa\DatasetPhiard, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=128, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_rat

## Statistik mean ± std + paired t-test

In [4]:
df = pd.DataFrame(results)
print('Raw results per seed:')
print(df.pivot(index='seed', columns='variant', values='top1').round(4))
print()

agg = df.groupby('variant').agg(
    top1_mean=('top1', 'mean'), top1_std=('top1', 'std'),
    top5_mean=('top5', 'mean'), top5_std=('top5', 'std'),
    time_mean=('time_s', 'mean'),
).round(4)
print('Agregat per variant:')
print(agg)

top1_attn   = df[df.variant == 'attn']['top1'].values
top1_noattn = df[df.variant == 'noattn']['top1'].values
diff = top1_attn - top1_noattn
t_stat, p_val = stats.ttest_rel(top1_attn, top1_noattn)

print(f'\nPaired t-test (attn - noattn):')
print(f'  mean diff = {diff.mean()*100:+.2f}%   std diff = {diff.std()*100:.2f}%')
print(f'  t-statistic = {t_stat:.3f}')
print(f'  p-value     = {p_val:.4f}   {"(SIGNIFIKAN p<0.05)" if p_val < 0.05 else "(tidak signifikan, p>=0.05)"}')

# Cohen's d effect size
pooled_std = np.sqrt((top1_attn.var(ddof=1) + top1_noattn.var(ddof=1)) / 2)
cohens_d = (top1_attn.mean() - top1_noattn.mean()) / pooled_std
print(f'  Cohen\'s d = {cohens_d:.3f} (effect size: {"small" if abs(cohens_d) < 0.5 else "medium" if abs(cohens_d) < 0.8 else "large"})')

Raw results per seed:
variant    attn  noattn
seed                   
7        0.8513  0.9588
42       0.9480  0.9731
123      0.9247  0.9713
999      0.9606  0.9337
2024     0.9140  0.9444

Agregat per variant:
         top1_mean  top1_std  top5_mean  top5_std  time_mean
variant                                                     
attn        0.9197    0.0425     0.9986    0.0032    60.0415
noattn      0.9563    0.0171     1.0000    0.0000    69.1953

Paired t-test (attn - noattn):
  mean diff = -3.66%   std diff = 4.32%
  t-statistic = -1.693
  p-value     = 0.1658   (tidak signifikan, p>=0.05)
  Cohen's d = -1.129 (effect size: large)


## Plot perbandingan

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

labels = ['with C2PSA', 'without C2PSA']
means = [top1_attn.mean(), top1_noattn.mean()]
stds = [top1_attn.std(), top1_noattn.std()]
colors = ['mediumseagreen', 'tomato']

axes[0].bar(labels, means, yerr=stds, capsize=8, color=colors, alpha=0.85,
            edgecolor='black', linewidth=1)
for i, vals in enumerate([top1_attn, top1_noattn]):
    axes[0].scatter([i] * len(vals), vals, color='black', zorder=10, s=40, alpha=0.7)
axes[0].set_ylabel('Top-1 accuracy')
axes[0].set_title(f'Mean ± std (n={len(SEEDS)} seeds)\np-value = {p_val:.4f}, Cohen\'s d = {cohens_d:.3f}')
axes[0].grid(axis='y', alpha=0.3)

for seed in sorted(SEEDS):
    a = df[(df.variant == 'attn') & (df.seed == seed)]['top1'].values[0]
    n = df[(df.variant == 'noattn') & (df.seed == seed)]['top1'].values[0]
    axes[1].plot([0, 1], [a, n], 'o-', label=f'seed={seed}', alpha=0.7)
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(labels)
axes[1].set_ylabel('Top-1 accuracy')
axes[1].set_title('Per-seed paired comparison')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle('Ablation Study: C2PSA Attention Block in YOLO26 — Aksara Jawa', fontsize=12)
plt.tight_layout()
plt.savefig(ROOT / 'runs' / 'ablation_c2psa_clean.png', dpi=120, bbox_inches='tight')
plt.show()

<Figure size 1300x500 with 2 Axes>

## Save results + summary buat paper

In [6]:
df.to_csv(ROOT / 'runs' / 'ablation_c2psa_clean_results.csv', index=False)

summary = f'''
=== Ablation Study: C2PSA in YOLO26-cls (Aksara Jawa) ===
Seeds: {SEEDS}

Variant A (with C2PSA):    {top1_attn.mean()*100:.2f}% ± {top1_attn.std()*100:.2f}%   ({2_812_104:,} params)
Variant B (without C2PSA): {top1_noattn.mean()*100:.2f}% ± {top1_noattn.std()*100:.2f}%   ({2_908_488:,} params)

Mean diff (attn - noattn): {diff.mean()*100:+.2f}%
Paired t-test: t={t_stat:.3f}, p={p_val:.4f}
Cohen's d: {cohens_d:.3f}

KESIMPULAN: {("H0 ditolak — C2PSA memberikan peningkatan signifikan" if p_val < 0.05 and diff.mean() > 0 else ("H0 ditolak — variant tanpa C2PSA signifikan lebih baik" if p_val < 0.05 and diff.mean() < 0 else "H0 diterima — TIDAK ada perbedaan signifikan, attention block tidak meaningful untuk task ini"))}
'''
print(summary)
(ROOT / 'runs' / 'ablation_c2psa_clean_summary.txt').write_text(summary, encoding='utf-8')


=== Ablation Study: C2PSA in YOLO26-cls (Aksara Jawa) ===
Seeds: [42, 123, 7, 2024, 999]

Variant A (with C2PSA):    91.97% ± 3.80%   (2,812,104 params)
Variant B (without C2PSA): 95.63% ± 1.53%   (2,908,488 params)

Mean diff (attn - noattn): -3.66%
Paired t-test: t=-1.693, p=0.1658
Cohen's d: -1.129

KESIMPULAN: H0 diterima — TIDAK ada perbedaan signifikan, attention block tidak meaningful untuk task ini



411

## Ringkasan hasil ablation

Pada split bersih aksara Jawa, menghapus blok C2PSA menaikkan rata-rata akurasi dari 91.97% menjadi 95.63% (selisih 3.66%). Paired t-test tidak signifikan (p = 0.166) karena varian variant attention tinggi, tetapi Cohen's d = -1.13 menunjukkan efek praktis yang besar ke arah konfigurasi tanpa attention. Variant tanpa attention menang pada 4 dari 5 seed. Pembahasan naratif lengkap ada di paper.
